# 💻 Notebook do Aluno — Aula 05: Embeddings e busca semântica com ChromaDB

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 05/14 — Módulo 2: RAG**  
**⏱️ 1h40min**  
**🔢 nomic-embed-text · ChromaDB**  
**🔁 Andaime 45%**  

---

## 🎯 Objetivo da aula

Entender como texto vira número e como números permitem encontrar documentos por significado — não por palavras-chave. Ao final, o grupo tem uma coleção ChromaDB do domínio pronta para o pipeline RAG da Aula 06.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime da aula.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain-ollama langchain-community chromadb numpy -q

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# 👉 LACUNA 1: instancie OllamaEmbeddings com o modelo correto
embeddings = OllamaEmbeddings(model=___)

# 👉 LACUNA 2: crie 10+ documentos do domínio do grupo
# Use Document(page_content="...", metadata={"categoria": "..."})
documentos = [
    Document(page_content=___, metadata={"categoria": ___}),
    # ... mais 9+ documentos
]

# 👉 LACUNA 3: crie o vector store com from_documents
db = Chroma.from_documents(
    documents=___,
    embedding=___,
    persist_directory="/content/chroma_ckp02",
)

# 👉 LACUNA 4: faça 5 queries semânticas e documente a qualidade
queries = [___, ___, ___, ___, ___]
for q in queries:
    res = db.similarity_search_with_score(q, k=3)
    print(f"\nQuery: {q}")
    for doc, score in res:
        print(f"  [{score:.3f}] {doc.page_content}")

# Converter em retriever para a Aula 06
retriever = db.as_retriever(search_kwargs={"k":3})
print(f"\nRetriever pronto para Aula 06: {type(retriever)}")

---

## ✍️ Suas anotações

Registre aqui suas observações sobre o andaime e os exercícios (qualidade dos resultados, comparações e conclusões).

---

## 🏋️ Exercícios da Aula 05

Quatro exercícios práticos sobre o conteúdo da aula — o cliente de embeddings com `nomic-embed-text`, a similaridade cosseno com numpy, o `metadata filtering` no ChromaDB e a leitura do score como DISTÂNCIA cosseno.

Rode no Google Colab, na ordem, com o domínio do grupo: use os documentos e metadados criados no andaime acima — eles alimentam o RAG da Aula 07.


### Exercício 1 — Cliente de embeddings · ★★☆ · 10 min

*Individual · Colab*

1. Complete o cliente de embeddings: o modelo do semestre em `model=___`.
2. Preencha a input em `embed_query(___)` com uma frase do seu domínio e rode: anote as dimensões e o tipo de cada valor.
3. Complete `embed_documents(___)` com 2–3 textos e confirme as dimensões de cada vetor.
4. Rode de novo com a mesma frase: o vetor muda ou é determinístico?

> **💡 Dica:** embedding não é um LLM de texto — `embed_query` recebe uma string e devolve uma lista de 768 floats.


In [ ]:
# Exercício 1 — cliente de embeddings: texto → vetor
from langchain_ollama import OllamaEmbeddings
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# 👉 LACUNA 1: o modelo de embedding do semestre (Slide 08)
embeddings = OllamaEmbeddings(model=___)

# 👉 LACUNA 2: gere o vetor de UMA frase — a string é a input do cliente
vetor = embeddings.embed_query(___)
print(f"Dimensões: {len(vetor)} · tipo de cada valor: {type(vetor[0]).__name__}")
print(f"Primeiros 5 valores: {vetor[:5]}")

# 👉 LACUNA 3: gere os vetores de vários documentos de uma vez
docs = ["Pizza com queijo", "Macarrão ao sugo", "Motor de carro"]
vetores = embeddings.___(docs)
print(f"Documentos: {len(vetores)} · dims de cada: {len(vetores[0])}")


### Exercício 2 — Similaridade cosseno com numpy · ★★☆ · 10 min

*Individual · Colab*

1. Complete a fórmula na função: `dot(a, b) / (norm(a) · norm(b))`.
2. Compare `comida italiana` com a frase do domínio e anote o valor.
3. Compare com a frase sem relação e veja a diferença entre os dois pares.
4. Confira a régua: ~1.0 = mesmo significado · ~0.5 = relação parcial · ~0.0 = sem relação.

> **💡 Dica:** `np.dot` e `np.linalg.norm` resolvem a conta inteira — sobra só a divisão.


In [ ]:
# Exercício 2 — similaridade cosseno com numpy (Slide 09)
import numpy as np

def similaridade_cosseno(v1, v2) -> float:
    a, b = np.array(v1), np.array(v2)
    # 👉 LACUNA 1: complete a fórmula — dot(a, b) / (norm(a) * norm(b))
    return float(np.dot(a, b) / (___ * ___))

vetor_query = embeddings.embed_query("comida italiana")
vetor_doc   = embeddings.embed_query("Pizza napolitana com mozarela")
vetor_fora  = embeddings.embed_query("Motor do carro faz barulho")

# 👉 LACUNA 2: calcule os dois pares e compare os valores
print(f"comida × pizza : {similaridade_cosseno(vetor_query, ___):.4f}")
print(f"comida × motor : {similaridade_cosseno(vetor_query, ___):.4f}")


### Exercício 3 — Metadata filtering no ChromaDB · ★★☆ · 10 min

*Individual · Colab*

1. Rode a query SEM filtro (`k=3`) e anote os resultados.
2. Complete o filtro simples com a chave de metadado que você criou no lab.
3. Complete o filtro composto com o operador `$and` (dois metadados simultâneos).
4. Compare: quem entrou, quem saiu — e por quê.

> **💡 Dica:** o `filter` restringe o espaço de busca ANTES do k-NN — um documento com alta similaridade pode ser excluído só por não bater com o metadado.


In [ ]:
# Exercício 3 — busca pura vs. metadata filtering (usa o db do lab)
query = "prato com carne"   # troque pela query do seu domínio

# 👉 LACUNA 1: rode a query SEM filtro — k=3
res_sem = db.similarity_search(query, k=___)
print("SEM filtro:", [d.page_content for d in res_sem])

# 👉 LACUNA 2: repita COM o filtro — use a chave de metadado do lab (ex.: categoria)
res_com = db.similarity_search(query, k=3, filter=___)
print("COM filtro:", [d.page_content for d in res_com])

# 👉 LACUNA 3: repita com o filtro composto $and (dois metadados simultâneos)
res_and = db.similarity_search(query, k=3, filter={"$and": [___, ___]})
print("COM $and:", [d.page_content for d in res_and])


### Exercício 4 — Corrija a leitura do score · ★★☆ · 10 min

*Individual · Colab*

1. Rode a versão ERRADA (`score > 0.75`) e veja o que ela seleciona.
2. Complete a conversão: distância → similaridade.
3. Compare os dois resultados no console.
4. Anote em um comentário por que o score é DISTÂNCIA cosseno e não similaridade.

> **💡 Dica:** distância cosseno 0.0 = vetores idênticos — 0.10 é ótimo (similaridade 0.90) e 0.90 é péssimo.


In [ ]:
# Exercício 4 — o score é DISTÂNCIA cosseno (menor = mais similar)
resultados = db.similarity_search_with_score("comida italiana", k=5)

THRESHOLD = 0.75
print("Versão ERRADA (score > 0.75):")
for doc, score in resultados:
    if ___ > THRESHOLD:   # 👉 LACUNA 1: a condição errada — rode e veja o que volta
        print(f"  ❌ [dist={score:.3f}] {doc.page_content[:60]}")

print("\nVersão CORRIGIDA (1 - score > 0.75):")
for doc, score in resultados:
    similaridade = ___    # 👉 LACUNA 2: converta a distância em similaridade
    relevante = "✅" if similaridade > THRESHOLD else "⚠️"
    print(f"  {relevante} [{similaridade:.3f}] {doc.page_content[:60]}")


## 📚 Referências da aula

- Docs ChromaDB — Documentação oficial: collections, add, query, persist, metadata filtering. docs.trychroma.com
- Docs LangChain — OllamaEmbeddings e integração com ChromaDB. python.langchain.com/docs/integrations/vectorstores/chroma
- Modelo Nomic AI — nomic-embed-text: especificações, benchmarks e casos de uso. huggingface.co/nomic-ai/nomic-embed-text-v1
- Paper Mikolov, T. et al. — "Efficient Estimation of Word Representations in Vector Space." (Word2Vec, 2013) — a fundação conceitual dos embeddings modernos. arxiv.org/abs/1301.3781
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15: Representação distribuída — a base teórica dos embeddings e espaços vetoriais.

---

**→ Próxima Aula — Aula 06 · 14/09** — Pipeline RAG completo — load, split, embed, retrieve, generate
  
Conectar o retriever à chain LCEL. O LLM responde com base nos seus documentos.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*